# AQI estimation from pollutant readings for five Indian cities

Samuel Flores and Santiago Becerra — ETL G51, final delivery.

This notebook extends the earlier EDA and the validated PostgreSQL warehouse. It does not replace the three-source ETL. The operational question is: **as soon as a city's daily pollutant readings arrive, what is that day's AQI and its CPCB category?** The historical stream is a replay for an academic demonstration, not live environmental monitoring.

Run the Airflow ETL first. The training module reads `india_dw.air_quality_enriched`, the same dimensional view used by the static report and the Kafka producer. Configure `INDIA_DATABASE_URL` in the environment; do not paste credentials into this notebook.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
ROOT = Path(os.environ.get('INDIA_PROJECT_ROOT', Path.cwd())).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'final_ml.py').exists(), 'Set INDIA_PROJECT_ROOT to the project directory'
sys.path.insert(0, str(ROOT / 'src'))
from final_ml import (FEATURE_COLUMNS, read_warehouse, build_features, training_labels,
                      temporal_split, train, forecastability)
DATABASE_URL = os.environ['INDIA_DATABASE_URL']
ARTIFACTS = ROOT / 'artifacts'
source = read_warehouse(DATABASE_URL)

## 1. Can tomorrow's AQI be forecast from today's data?

Our first model tried to forecast next-day AQI from the previous day's AQI, pollutants, weather and calendar. It did no better than always predicting the average. Before choosing a model we therefore check whether the data contain any day-to-day memory or seasonality. The check uses the training period only (2015–2021).

In [ ]:
import matplotlib.pyplot as plt
evidence = forecastability(source)
display(pd.Series(evidence['aqi_lag1_autocorrelation_by_city'], name='AQI lag-1 autocorrelation').round(3).to_frame())
print('Monthly mean AQI range (points):', round(evidence['monthly_mean_aqi_range'], 1))
city = source[source.city_name.eq('Delhi')].sort_values('date')
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(city.aqi.iloc[:-1], city.aqi.iloc[1:], s=4, alpha=.3)
ax.set(title='Delhi: AQI today vs AQI tomorrow', xlabel='AQI on day t', ylabel='AQI on day t+1')
fig.tight_layout()

The autocorrelation is practically zero in every city and the monthly averages are almost flat: knowing today's AQI says nothing about tomorrow's. No model can forecast what the data do not contain, so a next-day forecast is not a defensible use case for this dataset.

The pollutants of the **same day**, however, determine the AQI. The official CPCB AQI is the maximum of piecewise sub-indices per pollutant, so the relationship is strong but non-linear. Estimating it immediately from the readings is useful operationally: a monitoring centre can publish an AQI value and category as soon as the sensors report, before the official calculation is released.

## 2. Features and target

Inputs: `city_name` and the nine pollutant readings of the day (`pm25`, `pm10`, `no_value`, `no2`, `nox`, `nh3`, `co`, `so2`, `o3`). Target: the observed AQI of that same city-day. The observed AQI and its category are never inputs. Missing, negative or non-finite readings and duplicate keys fail explicitly; nothing is imputed to hide an ETL problem.

In [ ]:
features = build_features(source)
labeled = training_labels(source)
display(pd.DataFrame({'rows': [len(source), len(features), len(labeled)]}, index=['warehouse', 'inference rows', 'labeled rows']))
display(labeled[['city_name', 'target_date'] + FEATURE_COLUMNS[1:] + ['target_aqi']].head())
assert 'aqi' not in FEATURE_COLUMNS and 'target_aqi' not in FEATURE_COLUMNS and 'aqi_bucket' not in FEATURE_COLUMNS

## 3. Evaluation design

All cities share the same date boundaries: training through 2021, validation during 2022 and final test during 2023–2024. We compare Ridge regression, HistGradientBoosting and Random Forest and select the lowest validation MAE. The chosen model is refitted on training plus validation; the test period never chooses the model or fits preprocessing. A constant mean-AQI baseline and the linear model are reported as references. Besides MAE, RMSE and R², we report **category accuracy**: how often the estimated AQI falls in the same CPCB category as the observed one.

Airflow already trains and saves the model during the final DAG. By default this notebook reads those artifacts without changing the model used by the running Kafka consumer. Set `RETRAIN = True` to reproduce training from the current warehouse; then restart the consumer.

In [ ]:
training, validation, test = temporal_split(labeled)
display(pd.DataFrame([{'partition': name, 'rows': len(part), 'first_date': part.target_date.min(), 'last_date': part.target_date.max()} for name, part in [('training', training), ('validation', validation), ('test', test)]]))
assert training.target_date.max() < validation.target_date.min() < test.target_date.min()
RETRAIN = False
metadata_path = ARTIFACTS / 'metadata.json'
if RETRAIN or not metadata_path.exists():
    metadata = train(DATABASE_URL, ARTIFACTS)
else:
    metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    current_run_ids = sorted(source.run_id.astype(str).unique().tolist())
    saved_run_ids = sorted(str(value) for value in metadata.get('source_run_ids', []))
    if saved_run_ids != current_run_ids:
        raise ValueError('Saved model artifacts belong to another warehouse run. Set RETRAIN = True, then restart the consumer.')
display(pd.DataFrame(metadata['metrics']['validation']).T.round(3))
display(pd.DataFrame(metadata['metrics']['test']).T.round(3))
print('Selected:', metadata['model_name'], '| version:', metadata['model_version'])

In [ ]:
results = pd.read_csv(ARTIFACTS / 'test_predictions.csv', parse_dates=['target_date'])
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(results.target_aqi, results.predicted_aqi, s=4, alpha=.3)
axes[0].plot([0, 500], [0, 500], color='grey', lw=1)
axes[0].set(title='Test 2023-2024: observed vs estimated AQI', xlabel='Observed AQI', ylabel='Estimated AQI')
order = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
matrix = pd.crosstab(results.actual_category, results.predicted_category).reindex(index=order, columns=order, fill_value=0)
axes[1].imshow(matrix, cmap='Blues')
axes[1].set(xticks=range(6), yticks=range(6), title='CPCB category: observed (rows) vs estimated (columns)')
axes[1].set_xticklabels(order, rotation=30); axes[1].set_yticklabels(order)
for i in range(6):
    for j in range(6):
        axes[1].text(j, i, matrix.iat[i, j], ha='center', va='center', fontsize=8)
fig.tight_layout()

In [ ]:
from IPython.display import Markdown
m = metadata['metrics']['test']
display(Markdown(
    f"The selected **{metadata['model_name']}** estimates AQI on 2023–2024 with MAE **{m['selected_model']['mae']:.2f} AQI points** "
    f"and R² **{m['selected_model']['r2']:.3f}**, assigning the correct CPCB category in **{m['selected_model']['category_accuracy']:.1%}** of city-days. "
    f"Always predicting the mean gives MAE {m['mean_baseline']['mae']:.2f}; the linear model gives MAE {m['linear_reference']['mae']:.2f}, "
    "which confirms that the AQI formula is non-linear."))
display(Markdown('### Limitations\n' + '\n'.join('- ' + item for item in metadata['limitations'])))

## 4. Reuse during Kafka inference

The producer calls the same `build_features` function and publishes only the documented pollutant readings plus identifiers, the measurement date, the source run and the event creation time. The observed AQI stays offline. The consumer loads `artifacts/model.joblib` once, applies `predict` to each valid message and stores the estimate with the model version and Kafka position. The dashboard reads these consumer outputs and shows the estimated AQI and CPCB category; it does not retrain or query the warehouse.